<div style="background-color: #F4F6F7; padding: 20px; border-radius: 8px; font-family: 'Arial', sans-serif; color: #2E4053;">
  <!-- Logo centrado -->
  <div style="text-align: center; margin-bottom: 15px;">
    <img src="https://drive.google.com/uc?export=view&id=1kjzXfjTiieYAd4azw5bh4UfEg91gUIdh" alt="Logo Institucional" width="250" />
  </div>

  <!-- Título principal -->
  <h1 style="font-size: 36px; font-weight: bold; text-align: right;">
    <strong>Estimación del gradiente geotérmico a partir de imágenes multiespectrales del LANDSAT 7</strong>
  </h1>

  <!-- Datos del estudiante -->
  <p style="font-size: 22px; text-align: center; margin: 5px 0;">
    <strong>Pereira, 2025</strong>  &nbsp;|&nbsp;
    <strong>Institución:</strong> Universidad Tecnológica de Pereira  &nbsp;|&nbsp;
    <strong>Grupo de investigación en Automática </strong>
  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />

  <!-- Descripción breve -->
  <p style="font-size: 18px; text-align: center; font-style: italic; margin: 5px 0;">
    Este cuaderno es la primera prueba donde se entrena una red neuronal convolucional, la cual es una adaptación de ResNet-18 para regresión y, de esta manera, estimar el gradiente geotérmico a partir de imágenes multiespectrales. El código ncluye: ingestión y preprocesado de mosaicos (manejo de NoData), creación de muestras/parches, estrategia de fine-tuning, evaluación con métricas de regresión R2, MAE y RMSE y exportación de predicciones y checkpoints para análisis geoespacial.

  </p>

  <hr style="border: 1px solid #ABB2B9; margin: 10px 0;" />
</div>


# **1. Carga de las librerías, imágenes y la base de datos**

In [ ]:
!pip install -q --upgrade pip
!pip install -q rasterio==1.3.10 pyproj rioxarray
!pip install -q torchvision tqdm scikit-image pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 28.0 MB/s eta 0:00:00


In [ ]:
!gdown 1jxJkZG92IpRO8GWkLS3C3u_zk0ILLb45 # Base de datos
!gdown 1BAm52wXxOT0DeZBbgv9FgDrVCfMFrUF_ #Geotiffs

!unzip -q LANDSAT_GEOTIFF.zip -d LANDSAT_GEOTIFF

Downloading...
From: https://drive.google.com/uc?id=1jxJkZG92IpRO8GWkLS3C3u_zk0ILLb45
To: /content/normalized_data_multimodal2.csv
100% 1.18M/1.18M [00:00<00:00, 44.5MB/s]
Downloading...
From (original): https://drive.google.com/uc?id=1BAm52wXxOT0DeZBbgv9FgDrVCfMFrUF_
From (redirected): https://drive.google.com/uc?id=1BAm52wXxOT0DeZBbgv9FgDrVCfMFrUF_&confirm=t&uuid=2935e4f1-57e2-4e54-bec3-952e81f91d4e
To: /content/LANDSAT_GEOTIFF.zip
100% 7.69G/7.69G [00:42<00:00, 182MB/s]


# **Prueba con ResNet**

In [ ]:
#Librerías necesarias
import os
import glob
import math
import random
from typing import Optional, Callable, Tuple, List

import numpy as np
import pandas as pd
from tqdm import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, random_split
import torchvision.models as models
from sklearn.metrics import r2_score

try:
    from torchvision.models import ResNet18_Weights
except Exception:
    ResNet18_Weights = None

import rasterio

In [ ]:
# Reproducibilidad

def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

seed_everything(42)

In [ ]:
# Relleno de los valores faltantes con local_mean y nearest

class LandsatPatchDataset(Dataset):
    # Método 1 - FILL NEAREST
    def _fill_nodata_nearest_scipy(self, arr, mask):
        try:
            from scipy import ndimage as ndi
        except Exception:
            raise ImportError("Scipy no está instalado para nearest.")

        valid = mask.astype(bool)
        if valid.all():
            return arr.astype(np.float32)

        # Obtener índices del píxel válido más cercano
        _, inds = ndi.distance_transform_edt(~valid, return_indices=True)
        rows, cols = inds[0], inds[1]

        filled = np.empty_like(arr, dtype=np.float32)
        for b in range(arr.shape[0]):
            filled[b] = arr[b][rows, cols]

        return filled

    # Método 2 - FILL LOCAL_MEAN
        valid = mask.astype(bool).copy()
        H, W = valid.shape

        if valid.all():
            return arr.astype(np.float32)

        if max_iter is None:
            max_iter = max(H, W)

        arr_f = arr.astype(np.float32).copy()
        arr_f[:, ~valid] = 0.0

        bands = arr.shape[0]
        iter_count = 0

        while not valid.all() and iter_count < max_iter:

            pad_arr = np.pad(arr_f, ((0, 0), (1, 1), (1, 1)), mode='constant')
            pad_valid = np.pad(valid.astype(np.int32), ((1, 1), (1, 1)), mode='constant')

            neigh_sums = np.zeros_like(arr_f)
            neigh_counts = np.zeros((H, W), dtype=np.int32)

            # 8 vecinos
            neigh_sums += pad_arr[:, 0:H, 0:W]
            neigh_counts += pad_valid[0:H, 0:W]

            neigh_sums += pad_arr[:, 0:H, 1:W+1]
            neigh_counts += pad_valid[0:H, 1:W+1]

            neigh_sums += pad_arr[:, 0:H, 2:W+2]
            neigh_counts += pad_valid[0:H, 2:W+2]

            neigh_sums += pad_arr[:, 1:H+1, 0:W]
            neigh_counts += pad_valid[1:H+1, 0:W]

            neigh_sums += pad_arr[:, 1:H+1, 2:W+2]
            neigh_counts += pad_valid[1:H+1, 2:W+2]

            neigh_sums += pad_arr[:, 2:H+2, 0:W]
            neigh_counts += pad_valid[2:H+2, 0:W]

            neigh_sums += pad_arr[:, 2:H+2, 1:W+1]
            neigh_counts += pad_valid[2:H+2, 1:W+1]

            neigh_sums += pad_arr[:, 2:H+2, 2:W+2]
            neigh_counts += pad_valid[2:H+2, 2:W+2]

            can_fill = (~valid) & (neigh_counts > 0)
            if not np.any(can_fill):
                break

            for b in range(bands):
                sum_b = neigh_sums[b]
                arr_f[b, can_fill] = sum_b[can_fill] / neigh_counts[can_fill]

            valid[can_fill] = True
            iter_count += 1

        return arr_f

    # Contruir el dataset

    def __init__(self,
                 csv_path: str,
                 tiffs_dir: str,
                 target_size: Tuple[int, int] = (256, 256),
                 transform: Optional[Callable] = None,
                 nodata_fill: str = 'local_mean',
                 band_means: Optional[np.ndarray] = None,
                 add_mask_channel: bool = True,
                 mapper_row_to_file: Optional[Callable[[pd.Series, str], Optional[str]]] = None,
                 precomputed_paths: Optional[List[Optional[str]]] = None,
                 verbose: bool = False):

        self.df = pd.read_csv(csv_path, sep=None, engine='python')

        # Detectar lon/lat
        lon_cols = [c for c in self.df.columns if c.lower().startswith('lon')]
        lat_cols = [c for c in self.df.columns if c.lower().startswith('lat')]
        if not lon_cols or not lat_cols:
            raise ValueError("CSV debe contener columnas lon/lat")

        self.lon_col = lon_cols[0]
        self.lat_col = lat_cols[0]

        # Variable objetivo con "grad"
        tgt_cols = [c for c in self.df.columns if 'grad' in c.lower()]
        if not tgt_cols:
            raise ValueError("CSV debe contener columna con 'grad'")
        self.target_col = tgt_cols[0]

        # ID opcional
        self.id_col = None
        for cand in ['id', 'ID', 'point_id', 'identifier']:
            if cand in self.df.columns:
                self.id_col = cand
                break

        self.tiffs_dir = tiffs_dir
        self.target_h, self.target_w = target_size
        self.transform = transform
        self.nodata_fill = nodata_fill
        self.band_means = band_means
        self.add_mask_channel = add_mask_channel
        self.mapper_row_to_file = mapper_row_to_file
        self.verbose = verbose

        #  Buscar TIFFs
        if precomputed_paths is not None:
            if len(precomputed_paths) != len(self.df):
                raise ValueError("precomputed_paths debe coincidir con filas CSV")
            self.paths = list(precomputed_paths)
        else:
            all_tifs = []
            for root, _, files in os.walk(self.tiffs_dir):
                for f in files:
                    if f.lower().endswith(('.tif', '.tiff')):
                        all_tifs.append(os.path.join(root, f))

            def find_match(base):
                for p in all_tifs:
                    if os.path.basename(p) == base:
                        return p
                for p in all_tifs:
                    if base in os.path.basename(p):
                        return p
                return None

            possible_tif_cols = [c for c in self.df.columns if 'tif' in c.lower() or 'file' in c.lower()]
            self.paths = []

            for _, row in self.df.iterrows():
                path = None
                if possible_tif_cols:
                    cand = row[possible_tif_cols[0]]
                    if not pd.isna(cand):
                        s = str(cand)
                        if os.path.isabs(s) and os.path.exists(s):
                            path = s
                        else:
                            base = os.path.basename(s)
                            path = find_match(base)

                if path is None and mapper_row_to_file:
                    try:
                        cand = mapper_row_to_file(row, self.tiffs_dir)
                        if cand and os.path.exists(cand):
                            path = cand
                    except Exception:
                        pass

                if path is None and self.id_col:
                    idv = str(row[self.id_col])
                    for p in all_tifs:
                        if idv in os.path.basename(p):
                            path = p
                            break

                self.paths.append(path)

        if self.verbose:
            mapped = sum(1 for p in self.paths if p is not None)
            print(f"[Dataset] Total filas: {len(self.df)} | Mapeados: {mapped} | No mapeados: {len(self.df)-mapped}")

    def __len__(self):
        return len(self.df)

    # Lectura de los TIFF´s
    def _read_tiff(self, path: str):
        with rasterio.open(path) as src:
            arr = src.read(out_dtype='float32')  # (bands,H,W)
            nodata = src.nodata
            if nodata is None:
                nodata = np.nan

            if np.isnan(nodata):
                mask = ~np.isnan(arr).any(axis=0)
            else:
                mask = np.all(arr != nodata, axis=0)

        return arr, mask.astype(np.uint8)

    # Get Item
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        path = self.paths[idx]

        # Caso sin TIFF → devolver imagen negra
        if path is None or not os.path.exists(path):
            C = 1 + (1 if self.add_mask_channel else 0)
            dummy = np.zeros((C, self.target_h, self.target_w), dtype=np.float32)
            target = float(row[self.target_col])
            meta = {"id": row[self.id_col] if self.id_col else None,
                    "Longitude": row[self.lon_col],
                    "Latitude": row[self.lat_col],
                    "path": None}
            return torch.from_numpy(dummy), torch.tensor(target), meta

        arr, mask = self._read_tiff(path)
        arr = np.nan_to_num(arr, nan=0.0)

        # Relleno de los valores faltantes con local/nearest
        if self.nodata_fill == "nearest":
            try:
                arr = self._fill_nodata_nearest_scipy(arr, mask)
            except Exception:
                arr = self._fill_nodata_local_mean(arr, mask)

        elif self.nodata_fill == "local_mean":
            arr = self._fill_nodata_local_mean(arr, mask)

        elif self.nodata_fill == "band_mean" and self.band_means is not None:
            for b in range(arr.shape[0]):
                arr[b][mask == 0] = self.band_means[b]

        elif self.nodata_fill == "zero":
            arr[:, mask == 0] = 0.0

        # Añadir canal máscara
        if self.add_mask_channel:
            arr = np.concatenate([arr, mask[np.newaxis, :, :].astype(np.float32)], axis=0)

        t = torch.from_numpy(arr.astype(np.float32))
        t = F.interpolate(t.unsqueeze(0), size=(self.target_h, self.target_w),
                          mode='bilinear', align_corners=False).squeeze(0)

        target = float(row[self.target_col])
        meta = {
            "id": row[self.id_col] if self.id_col else None,
            "Longitude": row[self.lon_col],
            "Latitude": row[self.lat_col],
            "path": path
        }

        return t, torch.tensor(target, dtype=torch.float32), meta


# Calcular la media de las bandas

def compute_band_means(dataset: LandsatPatchDataset, sample_limit: Optional[int] = 500) -> np.ndarray:
    sums = None
    counts = None
    processed = 0
    n = min(len(dataset), sample_limit)

    for i in tqdm(range(n), desc="calc band means"):
        path = dataset.paths[i]
        if path is None or not os.path.exists(path):
            continue

        arr, mask = dataset._read_tiff(path)
        if arr.ndim == 2:
            arr = arr[np.newaxis, ...]

        valid = mask.astype(bool)
        if not valid.any():
            continue

        bsum = (arr * valid).sum(axis=(1, 2))
        bcount = np.full(arr.shape[0], valid.sum(), dtype=np.int64)

        if sums is None:
            sums = bsum.astype(np.float64)
            counts = bcount
        else:
            sums += bsum.astype(np.float64)
            counts += bcount

        processed += 1

    if processed == 0:
        raise RuntimeError("No se pudo calcular band_means: no se encontraron TIFF válidos.")

    return (sums / np.maximum(1, counts)).astype(np.float32)

In [ ]:
# Modelo ResNet

class ResNetRegressor(nn.Module):
    def __init__(self, in_channels: int, hidden: int = 256, pretrained: bool = False):
        super().__init__()
        # Construir resnet18 usando los pesos si es posible
        if pretrained and ResNet18_Weights is not None:
            res = models.resnet18(weights=ResNet18_Weights.DEFAULT)
        else:
            try:
                # weights=None para no preentrenar
                res = models.resnet18(weights=None)
            except Exception:
                # compat fallback
                res = models.resnet18(pretrained=False)

        # Se reemplaza la primera conv para aceptar el número de canales necesarios
        new_conv1 = nn.Conv2d(in_channels, 64, kernel_size=7, stride=2, padding=3, bias=False)
        try:
            if in_channels == 3:
                new_conv1.weight.data.copy_(res.conv1.weight.data)
            else:
                with torch.no_grad():
                    w = res.conv1.weight.data  # Tamaño (64,3,7,7)
                    avg = w.mean(dim=1, keepdim=True)  # (64,1,7,7)
                    # Repetir para los canales nuevos y escalar ligeramente
                    new_conv1.weight.data = avg.repeat(1, in_channels, 1, 1) * (3.0 / float(in_channels))
        except Exception:
            nn.init.kaiming_normal_(new_conv1.weight, mode='fan_out', nonlinearity='relu')

        res.conv1 = new_conv1

        # backbone sin los últimos bloques FC/avgpool
        self.backbone = nn.Sequential(*list(res.children())[:-2])
        feat_dim = 512  # resnet18 final feature dimension
        self.pool = nn.AdaptiveAvgPool2d((1, 1))
        self.flatten = nn.Flatten()
        self.shared = nn.Sequential(
            nn.Linear(feat_dim, hidden),
            nn.ReLU(),
            nn.Dropout(0.3)
        )
        self.head = nn.Linear(hidden, 1)

    def forward(self, x):
        f = self.backbone(x)
        f = self.pool(f)
        f = self.flatten(f)
        f = self.shared(f)
        out = self.head(f)
        return out.squeeze(1)

In [ ]:
# Helpers metas / predict
def unpack_metas(metas_batch):
    if metas_batch is None:
        return []

    # Si ya es lista/tupla de dicts
    if isinstance(metas_batch, (list, tuple)):
        if len(metas_batch) == 0:
            return []
        if isinstance(metas_batch[0], dict):
            return [dict(m) for m in metas_batch]
        # Si no son dicts, devolver dicts vacíos
        return [m if isinstance(m, dict) else {} for m in metas_batch]

    # Si es dict-of-lists (p.ej. DataLoader collate que devuelve dict de tensores)
    if isinstance(metas_batch, dict):
        keys = list(metas_batch.keys())
        first = metas_batch[keys[0]]
        try:
            n = len(first)
        except Exception:
            # Cada valor es escala, devolver un solo dict con valores simples
            return [{k: (v.item() if hasattr(v, 'item') else v) for k, v in metas_batch.items()}]

        out = []
        for i in range(n):
            d = {}
            for k in keys:
                v = metas_batch[k]
                try:
                    elem = v[i]
                    if hasattr(elem, 'item') and not isinstance(elem, str):
                        try:
                            d[k] = elem.item()
                        except Exception:
                            d[k] = elem
                    else:
                        d[k] = elem
                except Exception:
                    d[k] = v
            out.append(d)
        return out

    return []


def predict_on_loader(model, device, loader):
    model.eval()
    ys_list, preds_list, metas_all = [], [], []
    with torch.no_grad():
        for xb, yb, metas in loader:
            xb = xb.to(device)
            out = model(xb).cpu().numpy()
            ys_list.append(yb.numpy())
            preds_list.append(out)
            metas_list = unpack_metas(metas)
            metas_all.extend(metas_list)
    if len(ys_list) == 0:
        return np.array([]), np.array([]), metas_all
    ys = np.concatenate(ys_list, axis=0)
    preds = np.concatenate(preds_list, axis=0)
    return ys, preds, metas_all


def predict_and_save(model, device, loader, out_csv='predictions.csv'):
    model.eval()
    rows = []
    with torch.no_grad():
        for xb, yb, metas in tqdm(loader, desc="Predict"):
            xb = xb.to(device)
            preds = model(xb).cpu().numpy()
            yb_np = yb.numpy()
            metas_list = unpack_metas(metas)
            for i in range(len(preds)):
                m = metas_list[i] if i < len(metas_list) else {}
                rows.append({
                    'id': m.get('id') or m.get('ID') or None,
                    'Longitude': m.get('Longitude') or m.get('lon') or None,
                    'Latitude': m.get('Latitude') or m.get('lat') or None,
                    'grad_true': float(yb_np[i]) if yb_np is not None else None,
                    'grad_pred': float(preds[i]),
                    'path': m.get('path') if isinstance(m.get('path'), str) else None
                })
    df = pd.DataFrame(rows)
    df.to_csv(out_csv, index=False)
    print(f"[INFO] predictions saved to {out_csv}")
    return df

In [ ]:
# Ciclo de entrenamiento: MAE, RMSE, R2

def train_loop(model, device, train_loader, val_loader,
               epochs=30, lr=0.5e-4, save_path='model.pth',
               early_stop_patience: int = 8,
               save_val_preds_path: str = None):
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=3, factor=0.5)
    criterion = nn.MSELoss()
    best_val = float('inf')
    os.makedirs(os.path.dirname(save_path) or '.', exist_ok=True)
    no_improve = 0

    for epoch in range(1, epochs + 1):
        # Entrenamiento
        model.train()
        running_loss = 0.0
        n_samples = 0
        for xb, yb, _ in tqdm(train_loader, desc=f"Train E{epoch}", leave=False):
            xb = xb.to(device); yb = yb.to(device)
            preds = model(xb)
            loss = criterion(preds, yb)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            running_loss += loss.item() * xb.size(0)
            n_samples += xb.size(0)
        train_loss = running_loss / max(1, n_samples)

        # Validación
        model.eval()
        val_running = 0.0
        n_val = 0
        ys_all, preds_all = [], []
        with torch.no_grad():
            for xb, yb, _ in tqdm(val_loader, desc=f"Val E{epoch}", leave=False):
                xb = xb.to(device); yb = yb.to(device)
                out = model(xb)
                loss = criterion(out, yb)
                val_running += loss.item() * xb.size(0)
                n_val += xb.size(0)
                ys_all.append(yb.cpu().numpy()); preds_all.append(out.cpu().numpy())
        val_loss = val_running / max(1, n_val)

        if len(ys_all) > 0:
            ys_np = np.concatenate(ys_all, axis=0).ravel()
            preds_np = np.concatenate(preds_all, axis=0).ravel()
            mae = float(np.mean(np.abs(preds_np - ys_np)))
            rmse = float(np.sqrt(np.mean((preds_np - ys_np) ** 2)))
            try:
                r2 = r2_score(ys_np, preds_np)
            except Exception:
                r2 = float('nan')
        else:
            mae = float('nan'); rmse = float('nan'); r2 = float('nan')

        print(f"Epoch {epoch}: train_loss={train_loss:.6f} val_loss={val_loss:.6f} MAE={mae:.6f} RMSE={rmse:.6f} R2={r2:.4f}")

        scheduler.step(val_loss)

        # Guardar el mejor modelo
        if val_loss < best_val:
            best_val = val_loss
            no_improve = 0
            torch.save({'model_state': model.state_dict(), 'epoch': epoch, 'val_loss': val_loss}, save_path)
            print(f"[SAVE] best model saved to {save_path} (val_loss={val_loss:.6f})")
            if save_val_preds_path is not None:
                ys_v, preds_v, metas = predict_on_loader(model, device, val_loader)
                rows = []
                for i in range(len(preds_v)):
                    m = metas[i] if i < len(metas) else {}
                    rows.append({
                        'id': m.get('id') or m.get('ID') or None,
                        'Longitude': m.get('Longitude') or m.get('lon') or None,
                        'Latitude': m.get('Latitude') or m.get('lat') or None,
                        'grad_true': float(ys_v[i]) if ys_v.size > 0 else None,
                        'grad_pred': float(preds_v[i]) if preds_v.size > 0 else None,
                        'path': m.get('path') if isinstance(m.get('path'), str) else None
                    })
                pd.DataFrame(rows).to_csv(save_val_preds_path, index=False)
                print(f"[SAVE] validation predictions saved to {save_val_preds_path}")
        else:
            no_improve += 1

        if early_stop_patience and no_improve >= early_stop_patience:
            print(f"[EARLY STOP] no improvement for {no_improve} epochs. Stopping.")
            break

    return model

In [ ]:
# Bloque principal: rutas e hiperparámetros

if __name__ == "__main__":
    CSV_PATH = "/content/normalized_data_multimodal2.csv"   # CSV de la base de datos
    TIFFS_DIR = "/content/LANDSAT_GEOTIFF"                  # Carpeta los con GeoTIFFs
    TARGET_SIZE = (256, 256)  # Tamaño de las imágenes recortadas para su estándarización
    BATCH_SIZE = 16
    EPOCHS = 30
    LR = 0.2e-4
    MODEL_PATH = "/content/best_grad_model.pth"  # Mejor modelo obtenido
    PRED_CSV = "/content/predictions.csv"  # CSV de las predicciones sobre todos los datos
    TEST_PRED_CSV = "/content/test_predictions.csv"  # CSV de las predicciónes sobre test
    SAVE_VAL_PREDS = "/content/val_predictions_best.csv"  #CSV de las predicciones de validación sobre el mejor modelo
    ADD_MASK = True
    # Opciones de relleno: 'zero', 'band_mean', 'nearest', 'local_mean'
    NODATA_FILL = 'band_mean'
    SAMPLE_LIMIT_FOR_MEANS = 300
    TEST_SPLIT = 0.2
    VAL_SPLIT = 0.15
    NUM_WORKERS = 2
    SEED = 42

    seed_everything(SEED)

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print("Device:", device)

    # 1) build dataset (sin band_means inicialmente si vamos a calcularlas)
    ds = LandsatPatchDataset(csv_path=CSV_PATH, tiffs_dir=TIFFS_DIR,
                             target_size=TARGET_SIZE, nodata_fill=NODATA_FILL,
                             band_means=None, add_mask_channel=ADD_MASK, verbose=True)

    # Inspección rápida del mapping
    mapped = [p for p in ds.paths if p is not None]
    print("Mapped count:", len(mapped))
    print("First 10 mapped (sample):", mapped[:10])
    unmapped_count = sum(1 for p in ds.paths if p is None)
    print("Unmapped rows:", unmapped_count)
    if len(mapped) == 0:
        raise SystemExit(
            "Abortando: no se encontraron .tif mapeados. Verifica TIFFS_DIR, nombres y columnas del CSV."
        )

    # Cálculo de las medias de las bandas
    band_means = None
    if NODATA_FILL == 'band_mean':
        print("[INFO] Calculando la media de las bandas...")
        band_means = compute_band_means(ds, sample_limit=SAMPLE_LIMIT_FOR_MEANS)
        print("Band means:", band_means)
        # rebuild dataset con band_means
        ds = LandsatPatchDataset(csv_path=CSV_PATH, tiffs_dir=TIFFS_DIR,
                                 target_size=TARGET_SIZE, nodata_fill=NODATA_FILL,
                                 band_means=band_means, add_mask_channel=ADD_MASK, verbose=True)

    # 3) Split  de train / val / test
    n = len(ds)
    n_test = int(math.ceil(n * TEST_SPLIT))
    remaining = n - n_test
    n_val = int(math.ceil(remaining * VAL_SPLIT))
    n_train = remaining - n_val
    if n_train <= 0 or n_val < 0 or n_test < 0:
        raise ValueError(f"Split sizes invalid: n={n}, n_train={n_train}, n_val={n_val}, n_test={n_test}")

    # Reproducible split
    train_ds, val_ds, test_ds = random_split(ds, [n_train, n_val, n_test], generator=torch.Generator().manual_seed(SEED))
    print(f"train: {len(train_ds)}  val: {len(val_ds)}  test: {len(test_ds)}")

    pin_memory = True if torch.cuda.is_available() else False

    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True,
                              num_workers=NUM_WORKERS, pin_memory=pin_memory)
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False,
                            num_workers=NUM_WORKERS, pin_memory=pin_memory)
    test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False,
                             num_workers=NUM_WORKERS, pin_memory=pin_memory)

    # 4) Creación del modelo
    sample_x, _, _ = ds[0]
    in_ch = sample_x.shape[0]
    print("Model in_channels:", in_ch)
    model = ResNetRegressor(in_channels=in_ch, hidden=256, pretrained=False)
    model = model.to(device)

    # 5) Train
    model = train_loop(model, device, train_loader, val_loader,
                       epochs=EPOCHS, lr=LR, save_path=MODEL_PATH,
                       early_stop_patience=6, save_val_preds_path=SAVE_VAL_PREDS)

    # 6) Carga del mejor modelo y predecir sobre test y sobre todo el dataset
    if os.path.exists(MODEL_PATH):
        print(f"[INFO] Cargando mejor modelo desde {MODEL_PATH}")
        ckpt = torch.load(MODEL_PATH, map_location=device)
        model.load_state_dict(ckpt['model_state'])

        # Predicciones sobre test
        test_df = predict_and_save(model, device, test_loader, out_csv=TEST_PRED_CSV)
        print(f"[INFO] Las predicciónes de test se guardaron en {TEST_PRED_CSV} (rows: {len(test_df)})")

        # Predicciones sobre todo el dataset
        full_loader = DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False,
                                 num_workers=NUM_WORKERS, pin_memory=pin_memory)
        df_preds = predict_and_save(model, device, full_loader, out_csv=PRED_CSV)
        print(f"[INFO] Las predicciones de todo el conjunto de datos se guardaron en {PRED_CSV} (rows: {len(df_preds)})")
    else:
        print("[WARN] modelo no encontrado; no se generan predicciones.")

    print("Done.")

Device: cuda
[Dataset] Total filas: 4451 | Mapeados: 4451 | No mapeados: 0
Mapped count: 4451
First 10 mapped (sample): ['/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_2014.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_2014.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_4348.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_4348.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_3955.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_2360.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_4348.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_4348.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_3955.tif', '/content/LANDSAT_GEOTIFF/LANDSAT_GEOTIFF/LANDSAT7_3610.tif']
Unmapped rows: 0
[INFO] Calculando la media de las bandas...


calc band means: 100%|██████████| 300/300 [00:11<00:00, 26.14it/s]


Band means: [ 597.3492   724.35583  672.4746  3086.157   2102.3645  2444.8289
 2447.073   1071.9454 ]
[Dataset] Total filas: 4451 | Mapeados: 4451 | No mapeados: 0
train: 3026  val: 534  test: 891
Model in_channels: 9


Epoch 1: train_loss=375.729028 val_loss=162.653435 MAE=11.776363 RMSE=12.753565 R2=-5.6150
[SAVE] best model saved to /content/best_grad_model.pth (val_loss=162.653435)


[SAVE] validation predictions saved to /content/val_predictions_best.csv


...

Epoch 28: train_loss=18.767319 val_loss=18.877315 MAE=3.172701 RMSE=4.344803 R2=0.2323
[EARLY STOP] no improvement for 8 epochs. Stopping.
[INFO] Cargando mejor modelo desde /content/best_grad_model.pth


Predict: 100%|██████████| 56/56 [00:31<00:00,  1.77it/s]


[INFO] predictions saved to /content/test_predictions.csv
[INFO] Las predicciónes de test se guardaron en /content/test_predictions.csv (rows: 891)


Predict: 100%|██████████| 279/279 [02:40<00:00,  1.74it/s]

[INFO] predictions saved to /content/predictions.csv
[INFO] Las predicciones de todo el conjunto de datos se guardaron en /content/predictions.csv (rows: 4451)
Done.


In [ ]:
# Métricas de evaluación en test

from sklearn.metrics import r2_score
y_true = test_df["grad_true"].values
y_pred = test_df["grad_pred"].values

r2_test = r2_score(y_true, y_pred)
mae_test = float(np.mean(np.abs(y_pred - y_true)))
rmse_test = float(np.sqrt(np.mean((y_pred - y_true) ** 2)))

print(f"El R2 sobre test es: {r2_test:.4f}")
print(f"El MAE sobre test es: {mae_test:.4f}")
print(f"El RMSE sobre test es: {rmse_test:.4f}")

El R2 sobre test es: 0.3355
El MAE sobre test es: 3.0133
El RMSE sobre test es: 4.1341
